Cell 1 (code): GPU and environment check, then pinned install. If pip changes numpy, restart the runtime.

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| cuda", torch.version.cuda)


!pip install -q pycolmap==4.2.1

import pycolmap
print("pycolmap", pycolmap.__version__, "| has_cuda:", pycolmap.has_cuda)   # must be True


/bin/bash: line 1: nvidia-smi: command not found
3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
torch 2.14.1+cpu | cuda None


Cell 2 (code): confirm the install.

In [ ]:
import importlib.metadata as md
print("pycolmap", md.version("pycolmap"))


pycolmap 4.2.1


Cell 3 (code): print the real API. If cell 5 fails, paste me this output.

In [ ]:
import pycolmap
for fn in ["extract_features", "match_sequential", "match_exhaustive",
           "incremental_mapping", "undistort_images"]:
    print("=" * 20, fn); print(getattr(pycolmap, fn).__doc__)
print("ImageReaderOptions:", [m for m in dir(pycolmap.ImageReaderOptions) if not m.startswith("_")])
print("CameraMode:", list(pycolmap.CameraMode.__members__))
print("SequentialPairingOptions:", [m for m in dir(pycolmap.SequentialPairingOptions) if not m.startswith("_")])

==================== extract_features
extract_features(database_path: os.PathLike | str | bytes, image_path: os.PathLike | str | bytes, image_names: collections.abc.Sequence[str] = [], camera_mode: pycolmap._core.CameraMode = CameraMode.AUTO, reader_options: pycolmap._core.ImageReaderOptions = ImageReaderOptions(), extraction_options: pycolmap._core.FeatureExtractionOptions = FeatureExtractionOptions(), device: pycolmap._core.Device = Device.auto, cancellation_token: pycolmap._core.CancellationToken = None) -> None

Extract SIFT Features and write them to database

==================== match_sequential
match_sequential(database_path: os.PathLike | str | bytes, matching_options: pycolmap._core.FeatureMatchingOptions = FeatureMatchingOptions(), pairing_options: pycolmap._core.SequentialPairingOptions = SequentialPairingOptions(), verification_options: pycolmap._core.TwoViewGeometryOptions = TwoViewGeometryOptions(), device: pycolmap._core.Device = Device.auto, cancellation_token: pycolma

Cell 4 (code): mount Drive and import the module. Edit the three paths.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/Colab Notebooks")   # folder containing sfm_pipeline.py
import sfm_pipeline
importlib.reload(sfm_pipeline)

ROOT = "/content/drive/MyDrive/Colab Notebooks/MSDS688/Case Study"
VIDEO = "IMG_9175"            # or "IMG_9176"
FRAMES_SRC = f"{ROOT}/source/frames/{VIDEO}"
WORK = "/content/sfm_run"      # fast local Colab disk, not Drive

N_FRAMES     = None            # None = all frames
CAMERA_MODEL = "OPENCV"
OVERLAP      = 20
RUN_NAME = f"run_{N_FRAMES}frames_colab" if N_FRAMES else "run_full_colab"
DRIVE_OUT = f"{ROOT}/data/colmap/{VIDEO}/{RUN_NAME}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Cell 5 (code): run the pipeline. It runs extraction, sequential matching and incremental mapping, prints the registration summary, and writes all outputs to WORK.

In [ ]:
rec, summary = sfm_pipeline.run_sfm(
    FRAMES_SRC, WORK, n_frames=N_FRAMES, camera_model=CAMERA_MODEL, overlap=OVERLAP,
)

Using 40 frames
models found:            1  (registered per model: {0: 40})
best model id:           0
registered / total:      40 / 40
sparse 3D points:        23344
observations:            128257
mean reprojection error: 0.4691 px
unregistered frames (0): []


Cell 6 (code): save to Drive before the session ends.

In [ ]:
# import shutil
# from pathlib import Path
# dest = Path(DRIVE_OUT) / (f"run_{N_FRAMES}frames" if N_FRAMES else "run_full")
# shutil.copytree(WORK, dest, dirs_exist_ok=True)
# print("saved to", dest)
# !ls "{dest}"
import shutil
shutil.copytree(WORK, DRIVE_OUT, dirs_exist_ok=True)
print("saved to", DRIVE_OUT)
!ls "{DRIVE_OUT}"

saved to /content/drive/MyDrive/Colab Notebooks/colmap/run_40frames
database.db  sparse	  sparse_points.ply  undistorted
images	     sparse_best  summary.txt


WORK includes database.db and the copied frames. Delete those from Drive if space is tight.

Cell 7 (code): inspect the output layout. This shows how undistorted/ is arranged, which I haven't verified.

In [ ]:
!find /content/sfm_run -maxdepth 3 -not -name "*.jpg" | sort

/content/sfm_run
/content/sfm_run/database.db
/content/sfm_run/images
/content/sfm_run/sparse
/content/sfm_run/sparse/0
/content/sfm_run/sparse/0/cameras.bin
/content/sfm_run/sparse/0/frames.bin
/content/sfm_run/sparse/0/images.bin
/content/sfm_run/sparse/0/points3D.bin
/content/sfm_run/sparse/0/rigs.bin
/content/sfm_run/sparse_best
/content/sfm_run/sparse_best/cameras.bin
/content/sfm_run/sparse_best/frames.bin
/content/sfm_run/sparse_best/images.bin
/content/sfm_run/sparse_best/points3D.bin
/content/sfm_run/sparse_best/rigs.bin
/content/sfm_run/sparse_points.ply
/content/sfm_run/summary.txt
/content/sfm_run/undistorted
/content/sfm_run/undistorted/images
/content/sfm_run/undistorted/run-colmap-geometric.sh
/content/sfm_run/undistorted/run-colmap-photometric.sh
/content/sfm_run/undistorted/sparse
/content/sfm_run/undistorted/sparse/cameras.bin
/content/sfm_run/undistorted/sparse/frames.bin
/content/sfm_run/undistorted/sparse/images.bin
/content/sfm_run/undistorted/sparse/points3D.bin
